In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# 1. Student Configuration
STUDENT_ID = "TUPM-26-1261"
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())         # 29
RANDOM_SEED = int("".join(STUDENT_ID.split("-")[1:]))           # 261938

WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"

raw_measurements = np.genfromtxt(measurement_path, dtype=float)
labels = np.loadtxt(label_path, usecols=0, dtype=int)

n_features = raw_measurements.shape[1]
feature_names = [f"Feature_{i:03d}" for i in range(n_features)]

df = pd.DataFrame(raw_measurements, columns=feature_names)
df.insert(0, "Record_ID", np.arange(len(df)))  
df["Result"] = labels
df["Outcome"] = np.where(labels == -1, "Pass", "Fail")

without_all_missing = df[feature_names].dropna(axis=1, how="all")
usable_features = [
    col for col in without_all_missing.columns
    if without_all_missing[col].nunique(dropna=True) > 1
]

full_medians = df[usable_features].median()

rng = np.random.default_rng(seed=RANDOM_SEED)

pass_df = df[df["Outcome"] == "Pass"]
fail_df = df[df["Outcome"] == "Fail"]

per_group_n = min(20 + ID_SUM, len(pass_df), len(fail_df)) 

pass_idx = rng.choice(pass_df.index.to_numpy(), size=per_group_n, replace=False)
fail_idx = rng.choice(fail_df.index.to_numpy(), size=per_group_n, replace=False)

balanced_indices = np.concatenate([pass_idx, fail_idx])
balanced_df = df.loc[balanced_indices].copy().reset_index(drop=True)

selected_features = sorted(
    rng.choice(usable_features, size=5, replace=False).tolist()
)

balanced_df[selected_features] = balanced_df[selected_features].fillna(
    full_medians[selected_features]
)

export_cols = ["Record_ID", "Result", "Outcome"] + selected_features
balanced_df[export_cols].to_csv(OUTPUT_DIR / "balanced_sample.csv", index=False)

grouped_stats = balanced_df.groupby("Outcome")[selected_features].agg(
    ["count", "mean", "median", "std"]  # pandas .std() uses ddof=1 by default
)

comparison_rows = []

for feat in selected_features:
    n_pass = grouped_stats.loc["Pass", (feat, "count")]
    mean_pass = grouped_stats.loc["Pass", (feat, "mean")]
    med_pass = grouped_stats.loc["Pass", (feat, "median")]
    sd_pass = grouped_stats.loc["Pass", (feat, "std")]

    n_fail = grouped_stats.loc["Fail", (feat, "count")]
    mean_fail = grouped_stats.loc["Fail", (feat, "mean")]
    med_fail = grouped_stats.loc["Fail", (feat, "median")]
    sd_fail = grouped_stats.loc["Fail", (feat, "std")]

    abs_mean_diff = abs(mean_fail - mean_pass)

    denom_df = (n_pass - 1) + (n_fail - 1)
    if denom_df > 0:
        pooled_var = (((n_pass - 1) * (sd_pass ** 2)) + ((n_fail - 1) * (sd_fail ** 2))) / denom_df
        pooled_sd = np.sqrt(pooled_var)
    else:
        pooled_sd = 0.0
        
    if pooled_sd > 0:
        smd = (mean_fail - mean_pass) / pooled_sd
    else:
        smd = 0.0

    abs_smd = abs(smd)

    comparison_rows.append({
        "Feature": feat,
        "Pass_Count": int(n_pass),
        "Pass_Mean": round(mean_pass, 4),
        "Pass_Median": round(med_pass, 4),
        "Pass_SD": round(sd_pass, 4),
        "Fail_Count": int(n_fail),
        "Fail_Mean": round(mean_fail, 4),
        "Fail_Median": round(med_fail, 4),
        "Fail_SD": round(sd_fail, 4),
        "Abs_Mean_Diff": round(abs_mean_diff, 4),
        "Pooled_SD": round(pooled_sd, 4),
        "Standardized_Mean_Diff": round(smd, 4),
        "Abs_Standardized_Mean_Diff": round(abs_smd, 4)
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values(
    by="Abs_Standardized_Mean_Diff", ascending=False
).reset_index(drop=True)

comparison_df.to_csv(OUTPUT_DIR / "feature_comparison.csv", index=False)

pass_vals_np = balanced_df.loc[balanced_df["Outcome"] == "Pass", selected_features].to_numpy()
fail_vals_np = balanced_df.loc[balanced_df["Outcome"] == "Fail", selected_features].to_numpy()

np_pass_means = pass_vals_np.mean(axis=0)
pd_pass_means = balanced_df.groupby("Outcome")[selected_features].mean().loc["Pass"].to_numpy()
stats_match = np.allclose(np_pass_means, pd_pass_means)

top_row = comparison_df.iloc[0]
print("=== EXERCISE 2: ASSESSMENT DATA SUMMARY ===")
print(f"Balanced Sample Size             : {len(balanced_df)} ({per_group_n} Pass, {per_group_n} Fail)")
print(f"Selected Features                : {selected_features}")
print(f"Highest-Ranked Feature           : {top_row['Feature']}")
print(f"Absolute Standardized Mean Diff  : {top_row['Abs_Standardized_Mean_Diff']:.4f}")
print(f"NumPy-Pandas Statistics Match    : {stats_match}")
print("\nFeature Comparison Table:")
print(comparison_df[["Feature", "Pass_Mean", "Fail_Mean", "Abs_Mean_Diff", "Pooled_SD", "Abs_Standardized_Mean_Diff"]].to_string(index=False))


=== EXERCISE 2: ASSESSMENT DATA SUMMARY ===
Balanced Sample Size             : 76 (38 Pass, 38 Fail)
Selected Features                : ['Feature_070', 'Feature_095', 'Feature_214', 'Feature_418', 'Feature_579']
Highest-Ranked Feature           : Feature_095
Absolute Standardized Mean Diff  : 0.4219
NumPy-Pandas Statistics Match    : True

Feature Comparison Table:
    Feature  Pass_Mean  Fail_Mean  Abs_Mean_Diff  Pooled_SD  Abs_Standardized_Mean_Diff
Feature_095     0.0000     0.0001         0.0000     0.0001                      0.4219
Feature_070   620.9102   618.0464         2.8639     9.0773                      0.3155
Feature_214     0.0918     0.0788         0.0130     0.0422                      0.3079
Feature_579     0.0150     0.0161         0.0010     0.0060                      0.1698
Feature_418   315.6828   308.3847         7.2981   270.7618                      0.0270
